In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import torch
import torch.nn as nn
import wandb
from accelerate.test_utils.testing import get_backend
from core.models import NoisyMLP, load_model_from_artifact
from core.callbacks import WandBCallback
from lightning.pytorch.loggers import WandbLogger
from lightning import Trainer
from core.data import MNISTDataModule
from lightning.pytorch.callbacks import EarlyStopping
device, n_devices, _ = get_backend()

torch.set_float32_matmul_precision("highest")

## Dropout regularization in MNIST

Train a model without dropout

In [ ]:
noiseless_model = NoisyMLP(dropout_rate=0.0, out_features=1)
wandb_logger = WandbLogger(project="inductive-bias", name="noiseless-mlp", log_model=True)
mnist = MNISTDataModule(batch_size=32)
trainer = Trainer(max_epochs=25,
                  logger=wandb_logger, 
                #   callbacks=[EarlyStopping(monitor="val/loss")]
                  )
trainer.fit(noiseless_model, mnist) 
wandb.finish()

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jrudoler/.cache/pypoetry/virtualenvs/inductive ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | loss_func | BCEWithLogitsLoss | 0      | train
1 | layers    | Sequential        | 150 K  | train
--------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=25` reached.


epoch,▁▁▁▁▁▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇████
train/loss,█▄▂▁▆▁▁▁▁▂▂▁▁▁▁▂▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█████
val/loss,▆▃▂▁▁▂▁▂▂▂▄▃▃▂▄▄▂▃▃▃█▄▄▅▃
epoch,24
train/loss,0.0
trainer/global_step,46874
val/loss,0.03689


Train a model with dropout, which we suspect is equivalent to some L2 regularization

In [ ]:
noisy_model = NoisyMLP(dropout_rate=0.3, out_features=1)  # Initialize the model with dropout
wandb_logger = WandbLogger(project="inductive-bias", name="noisy-mlp", log_model=True)
mnist = MNISTDataModule(batch_size=32)
trainer = Trainer(max_epochs=25,
                  logger=wandb_logger, 
                  callbacks=[EarlyStopping(monitor="val/loss")])
trainer.fit(noisy_model, mnist) 
wandb.finish()

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | loss_func | CrossEntropyLoss | 0      | train
1 | layers    | Sequential       | 150 K  | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


RuntimeError: 0D or 1D target tensor expected, multi-target not supported

In [ ]:
noisy_model = load_model_from_artifact(NoisyMLP, 'jhrudoler-penn/inductive-bias/model-sxjn7tc3:latest')
noiseless_model = load_model_from_artifact(NoisyMLP, 'jhrudoler-penn/inductive-bias/model-n7awb9ov:latest')
no_stopping_model = load_model_from_artifact(NoisyMLP, 'jhrudoler-penn/inductive-bias/model-yv15t2cq:v0')
mnist = MNISTDataModule(batch_size=32)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


In [ ]:
torch.nn.functional.softmax(noisy_model(torch.randn(2, 1, 28, 28)))

/var/folders/7h/662tdm8d6sn0krrht717wzmm0000gq/T/ipykernel_82092/550448832.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  torch.nn.functional.softmax(noisy_model(torch.randn(2, 1, 28, 28)))


tensor([[1.],
        [1.]], grad_fn=<SoftmaxBackward0>)

In [ ]:
from core.bias import RidgeBias, BiasWithCrossEntropy, BiasWithBCE
module = BiasWithBCE(
        predictive_model=noisy_model,
        bias_model=RidgeBias(),
        grad_match_loss_fn=nn.functional.mse_loss,
        optimizer_cls=torch.optim.Adam,
        lr=1e-3,
    )

# Train using the Trainer interface.
trainer = Trainer(
    max_epochs=10,
    logger=WandbLogger(project="inductive-bias", name="ridge-bias-noisy"),
    callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss")],
    )
trainer.fit(module, mnist)

/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `v


  | Name             | Type      | Params | Mode 
-------------------------------------------------------
0 | predictive_model | NoisyMLP  | 150 K  | train
1 | bias_model       | RidgeBias | 1      | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
/Users/jrudoler/Library/Caches/pypoetry/virtualenvs/inductive-bias-XFXDbqEv-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

bias/beta,█▄▃▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆█████
train/loss,█▇▄▄▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
bias/beta,-0.0
epoch,4
train/loss,0.0
trainer/global_step,9349


In [ ]:
module = BiasWithCrossEntropy(
        predictive_model=noiseless_model,
        bias_model=RidgeBias(),
        loss_fn=nn.functional.mse_loss,
        optimizer_cls=torch.optim.Adam,
        lr=1e-3,
    )

# Train using the Trainer interface.
trainer = Trainer(
    max_epochs=10,
    logger=WandbLogger(project="inductive-bias", name="ridge-bias-noiseless"),
    callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss")],
    )
trainer.fit(module, mnist)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
wandb: Currently logged in as: jhrudoler (jhrudoler-penn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type      | Params | Mode 
-------------------------------------------------------
0 | predictive_model | NoisyMLP  | 150 K  | train
1 | bias_model       | RidgeBias | 1      | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/torch/autograd/graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


bias/beta,█▆▄▃▃▂▂▁▁▁▂▁▁▂▁▁▁▁▁▁▁▂▁▂▁▁▁▁▁▁▁▂▂▁▂▁▂▁▁▁
epoch,▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆███████████
train/loss,█▂▂▂▂▃▃▂▄▄▂▃▂▂▁▂▃▁▁▁▂█▃▂▂▂▂▂▄▂▂▄▃▄▂▂▃▂▃▂
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
bias/beta,0.3799
epoch,3
train/loss,25.0396
trainer/global_step,7499


In [ ]:
module = BiasWithCrossEntropy(
    predictive_model=no_stopping_model,
    bias_model=RidgeBias(),
    loss_fn=nn.functional.mse_loss,
    optimizer_cls=torch.optim.Adam,
    lr=1e-3,
)
trainer = Trainer(
    max_epochs=10,
    logger=WandbLogger(project="inductive-bias", name="ridge-bias-no-stopping"),
    callbacks=[WandBCallback(), EarlyStopping(monitor="train/loss")],
    )
trainer.fit(module, mnist)

/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'predictive_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['predictive_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'bias_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['bias_model'])`.
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type      | Params | Mode 
-------------------------------------------------------
0 | predictive_model | NoisyMLP  | 150 K  | train
1 | bias_model       | RidgeBias | 1      | train
-------------------------------------------------------
150 K     Trainable params
0         Non-trainable params
150 K     Total params
0.601     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
/home/jrudoler/.cache/pypoetry/virtualenvs/inductive-bias-v8JhmPLZ-py3.12/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

bias/beta,▁▂▃▅▅▆▆▇▇▇▇▇▆████▇▆▇███▇▇███▇▆▇████▇█▇█▆
epoch,▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆████████
train/loss,▁█▁▁▁▃▂▃▃▂▂▂▁▂▁▄▁▄▂▁▂▂▅▃▂▂▃▁▂▂▂▁▁▁▂▂▄▁▁▄
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█████
bias/beta,1.27503
epoch,3
train/loss,6461.88672
trainer/global_step,7499


In [ ]:

def predictive_loss_grad(
        predictions: torch.Tensor, targets: torch.Tensor, loss_fn
    ) -> torch.Tensor:
        # Compute per-sample gradient of the loss function with respect to the model params
        loss = loss_fn(predictions, targets)
        # Compute the gradient of the loss with respect to the model predictions (dL/dy_hat)
        # because we're using chain rule.
        per_sample_grad = torch.autograd.grad(
            loss, predictions, retain_graph=True, create_graph=True
        )[0]
        return per_sample_grad

# unit tests

# Test the predictive_loss_grad function on mse loss
def test_predictive_loss_grad_mse():
    predictions = torch.tensor([[1.0], [2.0], [3.0]], requires_grad=True)
    targets = torch.tensor([[1.5], [2.5], [3.5]])
    loss_fn = nn.MSELoss(reduction="sum")
    
    # Compute the gradient
    grad = predictive_loss_grad(predictions, targets, loss_fn)
    
    # Expected gradient: 2 * (predictions - targets)
    expected_grad = 2 * (predictions - targets)
    
    assert torch.allclose(grad, expected_grad), f"Expected {expected_grad}, but got {grad}"

test_predictive_loss_grad_mse()

In [ ]:
# load models
noisy_model = NoisyMLP(out_features=1)
noisy_state_dict = torch.load("saved_models/noisy-mlp.pt")
noisy_model.load_state_dict(noisy_state_dict)
noisy_model.eval()  # Set the model to evaluation mode

noiseless_model = NoisyMLP(out_features=1)
noiseless_state_dict = torch.load("saved_models/noiseless-mlp.pt")
noiseless_model.load_state_dict(noiseless_state_dict)
noiseless_model.eval()  # Set the model to evaluation mode



NoisyMLP(
  (loss_func): BCEWithLogitsLoss()
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): Dropout(p=0.2, inplace=False)
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Dropout(p=0.2, inplace=False)
    (5): Linear(in_features=128, out_features=128, bias=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=128, bias=True)
    (9): ReLU()
    (10): Dropout(p=0.2, inplace=False)
    (11): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [ ]:
test_x = torch.randn(2, 1, 28, 28)
noisy_pred = noisy_model(test_x)  # Test the model to ensure it's working
noiseless_pred = noiseless_model(test_x)  # Test the model to ensure it's working

In [ ]:
noisy_pred

tensor([[-21.6854],
        [-20.3525]], grad_fn=<AddmmBackward0>)

In [ ]:
noiseless_pred

tensor([[-14.5025],
        [-13.9359]], grad_fn=<AddmmBackward0>)

In [ ]:
from core.data import WikiTextDataModule

In [ ]:
wiki = WikiTextDataModule(batch_size=32)

In [ ]:
wiki_train = wiki.train_dataloader()

In [ ]:
wiki_train.dataset[0]

{'text': ''}